In [ ]:
import pandas as pd
from dotenv import load_dotenv
import requests
import json
from openai import OpenAI
from typing import Optional
from pydantic import BaseModel
import os
from tqdm import tqdm
from filtering_utils import (
    SYSTEM_PROMPT, get_statement, StructuredExtractionOutput,
    STRUCTURED_EXTRACTION_SCHEMA, _char_to_token_idx, apply_correction, apply_manual_corrections
)


# Load the environment variable
load_dotenv(dotenv_path="open-router.env")

c:\Users\Faruk\miniconda3\envs\algoverse\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


True

In [17]:
# Code to Check Open-Router Usage

response = requests.get(
   url="https://openrouter.ai/api/v1/key",
   headers={
     "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"
   }
 )
 
print(json.dumps(response.json(), indent=2))

{
  "data": {
    "label": "sk-or-v1-cea...5de",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 187,
    "limit_reset": null,
    "limit_remaining": 1.476739600000002,
    "include_byok_in_limit": false,
    "usage": 185.5232604,
    "usage_daily": 17.53913,
    "usage_weekly": 104.930226,
    "usage_monthly": 185.5232604,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_36We5MhQE5jYxTCLhqJpYlW3aGo",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


#### Loading the Datasets

In [22]:
# Load in the Factual Datasets (F0 skipped — no CoT-generated CSV exists for it yet)
F0_train, F0_test = pd.read_csv("../CoT_datasets/raw/F0_train.csv"), pd.read_csv("../CoT_datasets/raw/F0_test.csv")
F1_train, F1_test = pd.read_csv("../CoT_datasets/raw/F1_train.csv"), pd.read_csv("../CoT_datasets/raw/F1_test.csv")
F2_train, F2_test = pd.read_csv("../CoT_datasets/raw/F2_train.csv"), pd.read_csv("../CoT_datasets/raw/F2_test.csv")
F3_train, F3_test = pd.read_csv("../CoT_datasets/raw/F3_train.csv"), pd.read_csv("../CoT_datasets/raw/F3_test.csv")
F4_train, F4_test = pd.read_csv("../CoT_datasets/raw/F4_train.csv"), pd.read_csv("../CoT_datasets/raw/F4_test.csv")
F5_train, F5_test = pd.read_csv("../CoT_datasets/raw/F5_train.csv"), pd.read_csv("../CoT_datasets/raw/F5_test.csv")

# Load in the Arithmatic Statements
A1_train, A1_test = pd.read_csv("../CoT_datasets/raw/A1_train.csv"), pd.read_csv("../CoT_datasets/raw/A1_test.csv")
A2_train, A2_test = pd.read_csv("../CoT_datasets/raw/A2_train.csv"), pd.read_csv("../CoT_datasets/raw/A2_test.csv")
A3_train, A3_test = pd.read_csv("../CoT_datasets/raw/A3_train.csv"), pd.read_csv("../CoT_datasets/raw/A3_test.csv")

### Validating the Filtering Accuracy with a Manually Labelled and Checked Datasets

In [16]:
# Load in the Validaition Sets

F5_validation_set = pd.read_csv("../CoT_datasets/raw/F5_validated_150.csv")
A3_validation_set = pd.read_csv("../CoT_datasets/raw/A3_validated_50.csv")

In [17]:
F5_validation_set.head()

,generated_statement_ids,generated_statement_texts,extracted_statement_ids,extracted_statement_texts,prompt_len,is_extraction_point_correct,new_extraction_word,valence_laden_bias
0,"[128000, 128011, 66457, 220, 20, 315, 279, 276...",<｜User｜>Exactly 5 of the following cities are ...,"[128000, 128011, 66457, 220, 20, 315, 279, 276...",<｜User｜>Exactly 5 of the following cities are ...,177,False,would mean the statement is,False
1,"[128000, 128011, 66457, 220, 15, 315, 279, 276...",<｜User｜>Exactly 0 of the following cities are ...,"[128000, 128011, 66457, 220, 15, 315, 279, 276...",<｜User｜>Exactly 0 of the following cities are ...,169,False,0 are in Haiti (which is,False
2,"[128000, 128011, 66457, 220, 18, 315, 279, 276...",<｜User｜>Exactly 3 of the following cities are ...,"[128000, 128011, 66457, 220, 18, 315, 279, 276...",<｜User｜>Exactly 3 of the following cities are ...,174,False,"Therefore, the statement",False
3,"[128000, 128011, 66457, 220, 15, 315, 279, 276...",<｜User｜>Exactly 0 of the following cities are ...,"[128000, 128011, 66457, 220, 15, 315, 279, 276...",<｜User｜>Exactly 0 of the following cities are ...,175,False,both parts of the statement,False
4,"[128000, 128011, 66457, 220, 16, 315, 279, 276...",<｜User｜>Exactly 1 of the following cities are ...,"[128000, 128011, 66457, 220, 16, 315, 279, 276...",<｜User｜>Exactly 1 of the following cities are ...,168,False,"in Colombia, that part is",False


In [ ]:
def evaluate_sample(sample: pd.Series, model: str, max_retries: int = 3) -> (str | None):
    client = OpenAI(base_url="https://openrouter.ai/api/v1",
                    api_key=os.getenv("OPENROUTER_API_KEY"))

    last_error = None
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.parse(
                model=model,
                messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"""Statement: {get_statement(sample["generated_statement_texts"])}

    LLM Generated Output:
    {sample["generated_statement_texts"]}

    Extracted Text:
    {sample["extracted_statement_texts"]}

    Determine whether the extraction is correct."""
                }
            ],
                response_format=StructuredExtractionOutput,
                temperature=0
            )

            return response.choices[0].message.content
        except Exception as e:
            last_error = e
            print(f"evaluate_sample attempt {attempt + 1}/{max_retries} failed: {e!r}")

    raise last_error

#### Validation run for the Manually Labelled F5 Set

In [ ]:
count = 0
total = 50

for idx, row in F5_validation_set.head(total).iterrows():
    correction = json.loads(evaluate_sample(row, "openai/o3"))
    if correction["extraction_point_is_correct"] == row["is_extraction_point_correct"]:
        count += 1
    else:
        print(f"Mismatch # {idx}, Correct Label: {row["is_extraction_point_correct"]}, Model Label: {correction["extraction_point_is_correct"]}"
              f"\nExtracted Text: {row["extracted_statement_texts"]}"
              f"\nnew_pre_final_judgement_phrase: {correction["new_pre_final_judgement_phrase"]}")


print(f"{count}/{total} matched")

Mismatch # 5, Correct Label: False, Model Label: True
Extracted Text: <｜User｜>Exactly 5 of the following cities are in South Africa and 1 in Jamaica: Soweto, Benoni, Gqeberha, Thembisa, Hohhot, Soshanguve.

Work through the statement above before answering. State the facts you need — for each city named, where it actually is; for each arithmetic sub-expression, its actual value — then explicitly compute or count what the statement claims, and compare.

Do not use the words True, False, correct, incorrect, accurate, or inaccurate while reasoning. Reserve them for your conclusion only.

End your reasoning with exactly one line, either:
The statement is true.
or
The statement is false.
Then write </think>. After </think>, on its own line, write exactly one of:
Answer: Yes
or
Answer: No<｜Assistant｜><think>
Okay, so I have this problem here that says exactly 5 out of the listed cities are in South Africa and 1 in Jamaica. The list given is Soweto, Benoni, Gqeberha, Thembisa, Hohhot, and Sos

#### Validation run for the Two Mismatches from the F5 run

In [ ]:
# Rechecking the mismatchs with the new prompt setup
count = total = 0

for idx, row in F5_validation_set.iloc[[5, 22]].iterrows():
    total += 1
    correction = json.loads(evaluate_sample(row, "openai/o3"))
    if correction["extraction_point_is_correct"] == row["is_extraction_point_correct"]:
        count += 1
    else:
        print(f"Mismatch # {idx}, Correct Label: {row["is_extraction_point_correct"]}, Model Label: {correction["extraction_point_is_correct"]}"
              f"\nExtracted Text: {row["extracted_statement_texts"]}"
              f"\nnew_pre_final_judgement_phrase: {correction["new_pre_final_judgement_phrase"]}")


print(f"{count}/{total} matched")

2/2 matched


#### Validation run for the Manually Labelled A3 Set

In [ ]:
count = 0
total = 0

for idx, row in A3_validation_set.iterrows():
    total += 1
    correction = json.loads(evaluate_sample(row, "openai/o3"))
    if correction["extraction_point_is_correct"] == row["is_extraction_point_correct"]:
        count += 1
    else:
        print(f"Mismatch # {idx}, Correct Label: {row["is_extraction_point_correct"]}, Model Label: {correction["extraction_point_is_correct"]}"
              f"\nExtracted Text: {row["extracted_statement_texts"]}"
              f"\nnew_pre_final_judgement_phrase: {correction["new_pre_final_judgement_phrase"]}")


print(f"{count}/{total} matched")

Mismatch # 7, Correct Label: False, Model Label: True
Extracted Text: <｜User｜>(8 - 32) + (5 - 2) = -26

Work through the statement above before answering. State the facts you need — for each city named, where it actually is; for each arithmetic sub-expression, its actual value — then explicitly compute or count what the statement claims, and compare.

Do not use the words True, False, correct, incorrect, accurate, or inaccurate while reasoning. Reserve them for your conclusion only.

End your reasoning with exactly one line, either:
The statement is true.
or
The statement is false.
Then write </think>. After </think>, on its own line, write exactly one of:
Answer: Yes
or
Answer: No<｜Assistant｜><think>
Okay, so I have this equation here: (8 - 32) + (5 - 2) equals -26. I need to figure out if that's true or not. Let me break it down step by step.

First, let's look at the left side of the equation: (8 - 32) + (5 - 2). There are two parts inside the parentheses, so I should handle each pa

### Evaluation of Validation Performance

One of these, spesifically the mismatch # 46, is a mislabel from the validation set. So the accuracy on this 100 sample validation set is 98 percent (98/100).

### Filtering Datasets

In [ ]:
from ast import literal_eval
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Llama-8B")


def filter_dataset(dataset, tokenizer, evaluator_model, dataset_file_path, save_every=100):
    if len(dataset) > 0 and isinstance(dataset["generated_statement_ids"].iloc[0], str):
        dataset["generated_statement_ids"] = dataset["generated_statement_ids"].apply(literal_eval)

    for i, (idx, row) in tqdm(enumerate(dataset.iterrows()), total=len(dataset)):
        if (i + 1) % save_every == 0:
            dataset.to_csv(dataset_file_path, index=False)
            tqdm.write(f"Checkpointed at row {i + 1}/{len(dataset)}")

        try:
            correction = json.loads(evaluate_sample(row, evaluator_model))
        except Exception as e:
            tqdm.write(f"Row {idx}: evaluate_sample failed after retries ({e!r}), skipping")
            continue

        apply_correction(dataset, idx, row, correction, tokenizer)

    dataset.to_csv(dataset_file_path, index=False)  # final save
    return dataset

#### Filtering F5 

In [ ]:
filtered_F5_test = filter_dataset(dataset=F5_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F5_filtered_test.csv")

  4%|▍         | 23/593 [03:35<1:37:31, 10.27s/it]

Row 22: phrase not found verbatim, skipping: 'Angola. That doesn\x19t'


  5%|▍         | 27/593 [04:05<1:12:36,  7.70s/it]

Row 26: phrase found multiple times (ambiguous), skipping: 'Nicaragua, which'


  7%|▋         | 39/593 [05:57<1:31:21,  9.89s/it]

...

Row 562: phrase found multiple times (ambiguous), skipping: 'cities in Ukraine, which'


 98%|█████████▊| 579/593 [1:18:36<01:58,  8.45s/it]

Row 578: phrase found multiple times (ambiguous), skipping: 'That'


100%|██████████| 593/593 [1:20:06<00:00,  8.11s/it]

Row 592: phrase found multiple times (ambiguous), skipping: 'the statement'


In [ ]:
filtered_F5_train = filter_dataset(dataset=F5_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F5_filtered_train.csv")

 14%|█▍        | 196/1383 [25:37<1:59:44,  6.05s/it]

Row 195: phrase not found verbatim, skipping: 'Angola. This'


 16%|█▌        | 217/1383 [28:03<2:17:02,  7.05s/it]

Row 216: phrase not found verbatim, skipping: 'either:\nThe statement is'


 21%|██▏       | 295/1383 [37:22<2:17:13,  7.57s/it]

...

 96%|█████████▌| 1321/1383 [2:51:30<06:32,  6.33s/it]

Row 1320: phrase not found verbatim, skipping: 'statement given. That'


 97%|█████████▋| 1341/1383 [2:54:21<05:43,  8.18s/it]

Row 1340: phrase not found verbatim, skipping: 'in Iraq). That'


100%|██████████| 1383/1383 [3:00:02<00:00,  7.81s/it]


In [ ]:
corrections = {
    195: '- Aihara: Japan\n\nThis',
    216: 'drop',  # DROP (truncated)
    294: 'Edogawa). That',
    304: "South Korea. That doesn't",
    310: '2. So that seems',
    381: "That doesn't",
    396: 'Mozambique. That',
    467: 'in Guinea. That',
    490: "This doesn't",
    587: 'drop',  # DROP (truncated)
    603: 'Macedonia. That',
    614: '1 city.\n\nThat',
    692: '(2 cities)\n\nThat',
    695: 'drop',  # DROP (truncated)
    778: 'Omsk)\n\nThat',
    791: 'cities. That',
    843: 'Jammu).\n\nThat',
    872: '2 in Vietnam. That',
    929: 'drop',  # DROP (truncated)
    1062: '2 cities.\n\nThat',
    1064: 'in Russia. That',
    1141: '(4 cities)\n\nThat',
    1181: "in Côte d'Ivoire. That",
    1282: 'Korea. That',
    1320: 'Konya). That',
    1340: 'Erbil). That',
}
filtered_F5_train = apply_manual_corrections(
    dataset=filtered_F5_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F5_filtered_train.csv",
)

Row 195: corrected.
Row 294: corrected.
Row 304: corrected.
Row 310: corrected.
Row 381: corrected.
Row 396: corrected.
Row 467: corrected.
Row 490: corrected.
Row 603: corrected.
Row 614: corrected.
Row 692: corrected.
Row 778: corrected.
Row 791: corrected.
Row 843: corrected.
Row 872: corrected.
Row 1062: corrected.
Row 1064: corrected.
Row 1141: corrected.
Row 1181: corrected.
Row 1282: corrected.
Row 1320: corrected.
Row 1340: corrected.
Dropping 4 row(s): [216, 587, 695, 929]


In [ ]:
corrections = {
    22: "Angola. That doesn't",  # valence
    26: 'in Nicaragua, which',
    38: 'Norway (which is',
    50: '1 city (Dakar)\n\nThat',
    67: '→ 3 cities.\n\nThat',
    72: 'in Guinea," seems',
    81: 'and Zarqa.\n\nThat',
    87: "Haiti, that doesn't",  # valence
    98: '→ 3 cities.\n\nThat',
    105: '→ 4 cities\n\nThat',
    107: 'in Vietnam. That',
    140: 'Pinas, Davao (4)\n\nThis',
    163: 'Burundy. Therefore, the statement is',
    167: 'drop',  # DROP
    180: 'Eritrea. The statement',
    204: '→ 4 cities.\n\nThat',
    228: 'in the DRC. That',
    269: 'means the statement is',
    278: '3 in Spain and 3 in Germany is',
    293: 'the statement is',
    315: '1 in Madagascar, which',
    371: "Wait, that doesn't",  # valence
    403: 'of the statement are',
    413: 'none are in Kenya is',
    415: 'Rajshahi, Rangpur). That',
    424: '→ 2 cities.\n\nThat',
    426: 'drop',  # DROP
    432: 'Angola. So this part is',
    433: 'exactly 3 in Mozambique is',
    439: 'cities listed, which',
    467: 'claiming something',
    484: 'in Morocco, the statement is',
    485: "one is. So that's",
    498: 'are, that part is',
    522: '0 are in the UAE is',
    529: 'drop',  # DROP
    539: 'six are. Therefore, the statement is',
    562: 'claiming exactly 2 is',
    578: 'has 3 cities. That',
    592: 'Since both parts are',
}

filtered_F5_test = apply_manual_corrections(
    dataset=filtered_F5_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F5_filtered_test.csv",
)

Row 22: corrected.
Row 26: corrected.
Row 38: corrected.
Row 50: corrected.
Row 67: corrected.
Row 72: corrected.
Row 81: corrected.
Row 87: corrected.
Row 98: corrected.
Row 105: corrected.
Row 107: corrected.
Row 140: corrected.
Row 163: corrected.
Row 180: corrected.
Row 204: corrected.
Row 228: corrected.
Row 269: corrected.
Row 278: corrected.
Row 293: corrected.
Row 315: corrected.
Row 371: corrected.
Row 403: corrected.
Row 413: corrected.
Row 415: corrected.
Row 424: corrected.
Row 432: corrected.
Row 433: corrected.
Row 439: corrected.
Row 467: corrected.
Row 484: corrected.
Row 485: corrected.
Row 498: corrected.
Row 522: corrected.
Row 539: corrected.
Row 562: corrected.
Row 578: corrected.
Row 592: corrected.
Dropping 3 row(s): [167, 426, 529]


#### Filtering F4

In [ ]:
filtered_F4_test = filter_dataset(dataset=F4_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F4_filtered_test.csv")

 12%|█▏        | 72/598 [09:21<1:07:08,  7.66s/it]

Row 71: phrase found multiple times (ambiguous), skipping: 'Therefore, the statement is'


 19%|█▊        | 111/598 [14:31<1:01:11,  7.54s/it]

Row 110: phrase found multiple times (ambiguous), skipping: 'the statement is'


 25%|██▌       | 152/598 [20:12<1:08:04,  9.16s/it]

...

 94%|█████████▍| 565/598 [1:15:14<04:18,  7.83s/it]

Row 564: phrase found multiple times (ambiguous), skipping: 'in Saudi Arabia, which'


100%|█████████▉| 596/598 [1:19:18<00:14,  7.29s/it]

Row 595: phrase found multiple times (ambiguous), skipping: 'the statement is'


100%|██████████| 598/598 [1:19:36<00:00,  7.99s/it]


In [ ]:
filtered_F4_train = filter_dataset(dataset=F4_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F4_filtered_train.csv")

  1%|          | 14/1394 [02:06<4:08:48, 10.82s/it]

Row 13: phrase found multiple times (ambiguous), skipping: 'Therefore, the statement is'


  2%|▏         | 21/1394 [02:55<2:39:02,  6.95s/it]

Row 20: phrase not found verbatim, skipping: 'So, the statement is'


  2%|▏         | 27/1394 [03:47<3:14:37,  8.54s/it]

...

 94%|█████████▍| 1310/1394 [3:57:35<16:06, 11.51s/it]

Row 1309: phrase not found verbatim, skipping: 'statement given. That'


 98%|█████████▊| 1369/1394 [4:09:08<06:18, 15.14s/it]

Row 1368: phrase found multiple times (ambiguous), skipping: 'in Kenya, the statement is'


100%|██████████| 1394/1394 [4:13:32<00:00, 10.91s/it]


In [ ]:
corrections = {
    71: 'only 3 are. Therefore, the statement is',
    110: 'that would mean the statement is',
    151: 'in Rwanda, which',
    235: 'That would mean the statement is',
    263: 'exactly one city is in Chad, which',  # flip/confusion -> resolved at 'which aligns with the statement' (final clean disclosure)
    281: 'that would mean the statement is',
    286: 'saying that exactly five (all) are in Vietnam, which is',  # flip incorrect->true -> first corrected disclosure 'which is true'
    319: 'none are, then the statement is',  # paraphrase disclosure 'is claiming something that isn't true' precedes explicit 'is false'; landed on copula 'is' (alt: explicit line)
    413: 'So that would mean the statement is',
    484: 'So that would mean the statement is',
    490: 'exactly three cities are in Poland',  # soft anchor 'as stated' (= just as claimed); alt: explicit 'the original statement is [true]'
    495: 'in Syria, which directly',  # verifier phrase not found verbatim; rebuilt at 'which directly contradicts'
    506: 'claiming exactly two are in Finland is',
    517: 'in Mauritania is',
    564: 'exactly four are in Saudi Arabia, which',  # flip/confusion (Sultanah) -> resolved at 'which matches the reality'
    595: 'actually there, the statement is',
}

filtered_F4_test = apply_manual_corrections(
    dataset=filtered_F4_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F4_filtered_test.csv",
)

Row 71: corrected.
Row 110: corrected.
Row 151: corrected.
Row 235: corrected.
Row 263: corrected.
Row 281: corrected.
Row 286: corrected.
Row 319: corrected.
Row 413: corrected.
Row 484: corrected.
Row 490: corrected.
Row 495: corrected.
Row 506: corrected.
Row 517: corrected.
Row 564: corrected.
Row 595: corrected.


In [ ]:
corrections = {
    13: 'only four are. Therefore, the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    20: 'only one is. Therefore, the statement is',  # rebuilt (verifier phrase not verbatim)
    26: 'That means the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    69: 'listed cities are in the UAE',  # soft anchor 'as stated' (= just as claimed); land before it (alt: later explicit verdict)
    121: 'in Japan is five, which',
    186: 'in the DRC. That',
    275: 'exactly three are in South Korea, which',
    290: 'So the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    294: 'in Ukraine. That',  # rebuilt (verifier phrase not verbatim)
    345: 'drop',  # DROP - truncated mid-reasoning, no </think>, no verdict
    417: 'one is. So that',
    432: 'drop',  # DROP - truncated mid-reasoning, no </think>, no verdict
    437: 'says exactly 5 are in Mexico, which is',
    570: 'claiming exactly 3 are in Suriname is',
    583: 'none of these cities are in Angola is',
    599: 'exactly four of them are in Bangladesh',  # soft anchor 'as stated' (= just as claimed); land before it (alt: later explicit verdict)
    613: 'in Tanzania. This',  # rebuilt (verifier phrase not verbatim)
    698: 'one of them is in Vietnam is',
    740: 'one of them is in Saudi Arabia is',
    752: 'would mean the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    754: 'exactly four cities are in Uzbekistan',  # soft anchor 'as stated' (= just as claimed); land before it (alt: later explicit verdict)
    763: 'would mean the statement is',
    804: 'only one city is in Afghanistan, making the statement',  # flip/confusion resolved late; cut at final in-reasoning disclosure
    818: 'which would mean the statement is',
    828: 'two of them are in Russia. That',  # rebuilt (verifier phrase not verbatim)
    829: 'according to my findings, the statement is',
    830: 'would mean the statement is',
    854: 'That means the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    874: 'in Malaysia. That',
    919: 'would mean the statement is',
    957: 'them are in Saudi Arabia is',
    998: 'are in Algeria" must be',  # rebuilt (verifier phrase not verbatim)
    1106: 'all 5 are there. So the statement is',  # paraphrase 'the statement is claiming [discrepancy]' taken as first disclosure; land on copula 'is' (alt: later explicit 'is false/true')
    1131: '3 are in Morocco, which',
    1155: 'only four are. Therefore, the statement is',
    1185: 'Since only one is, the statement is',
    1279: 'Dar es Salaam. So, that',
    1298: 'is not. That',  # rebuilt (verifier phrase not verbatim)
    1309: 'in Russia. That',  # rebuilt (verifier phrase not verbatim)
    1368: 'indeed in Kenya, the statement is',
}

filtered_F4_train = apply_manual_corrections(
    dataset=filtered_F4_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F4_filtered_train.csv",
)

Row 13: corrected.
Row 20: corrected.
Row 26: corrected.
Row 69: corrected.
Row 121: corrected.
Row 186: corrected.
Row 275: corrected.
Row 290: corrected.
Row 294: corrected.
Row 417: corrected.
Row 437: corrected.
Row 570: corrected.
Row 583: corrected.
Row 599: corrected.
Row 613: corrected.
Row 698: corrected.
Row 740: corrected.
Row 752: corrected.
Row 754: corrected.
Row 763: corrected.
Row 804: corrected.
Row 818: corrected.
Row 828: corrected.
Row 829: corrected.
Row 830: corrected.
Row 854: corrected.
Row 874: corrected.
Row 919: corrected.
Row 957: corrected.
Row 998: corrected.
Row 1106: corrected.
Row 1131: corrected.
Row 1155: corrected.
Row 1185: corrected.
Row 1279: corrected.
Row 1298: corrected.
Row 1309: corrected.
Row 1368: corrected.
Dropping 2 row(s): [345, 432]


#### Filtering F3

In [ ]:
filtered_F3_test = filter_dataset(dataset=F3_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F3_filtered_test.csv")

  1%|          | 4/600 [01:11<3:05:49, 18.71s/it]

Row 3: phrase found multiple times (ambiguous), skipping: 'So the statement is'


  1%|▏         | 8/600 [02:03<2:12:09, 13.39s/it]

Row 7: phrase found multiple times (ambiguous), skipping: 'then the statement is'


 13%|█▎        | 79/600 [24:27<3:15:15, 22.49s/it]

...

 95%|█████████▌| 572/600 [2:18:37<07:16, 15.59s/it]

Row 571: phrase found multiple times (ambiguous), skipping: 'the statement is'


 96%|█████████▌| 575/600 [2:19:31<07:09, 17.17s/it]

Row 574: phrase found multiple times (ambiguous), skipping: 'in Brazil, the statement would be'


100%|██████████| 600/600 [2:24:49<00:00, 14.48s/it]


In [ ]:
filtered_F3_train = filter_dataset(dataset=F3_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F3_filtered_train.csv")

  1%|          | 12/1398 [02:16<3:48:14,  9.88s/it]

Row 11: phrase found multiple times (ambiguous), skipping: 'would be'


  1%|▏         | 20/1398 [03:58<5:50:00, 15.24s/it]

Row 19: phrase found multiple times (ambiguous), skipping: 'Tonga, which'


  5%|▌         | 71/1398 [16:58<7:25:13, 20.13s/it]

...

 98%|█████████▊| 1372/1398 [4:30:45<05:07, 11.81s/it]

Row 1371: phrase found multiple times (ambiguous), skipping: 'the statement is'


 99%|█████████▉| 1383/1398 [4:32:11<02:04,  8.30s/it]

Row 1382: phrase found multiple times (ambiguous), skipping: 'the statement is'


100%|██████████| 1398/1398 [4:34:01<00:00, 11.76s/it]


In [ ]:
corrections = {
    3: 'number is two, which',  # match via 'which matches'; land 'which' (P=quote/paraphrase anchor)
    7: 'drop',  # DROP - truncated mid-reasoning, no </think>, no verdict
    78: 'only Pokhara is, making the statement',  # 'making the statement [true/false]'; land on token before the verdict word
    79: 'the statement that exactly one is in Iraq',  # flip (Iran->Iraq); cut at corrected verdict 'making the statement ... incorrect'; land last claim token before 'incorrect'
    95: 'Morocco, then the statement is',
    113: 'making the statement that exactly two are in China',  # 'making the statement [true/false]'; land on token before the verdict word
    126: 'none of them would be, making the statement',  # 'making the statement [true/false]'; land on token before the verdict word
    131: 'in Armenia, which is',  # 'which is true'; land 'is'
    174: 'in South Korea. So the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at final in-reasoning commit
    179: 'is zero. The statement',  # 'The statement correctly asserts' (correctly=accurately); land 'statement', excl 'correctly'
    232: 'but Matola is, that',  # 'that fits the condition' (=matches); land 'that'
    273: 'drop',  # DROP - truncated mid-reasoning, no </think>, no verdict
    333: 'one of them is in the DRC, the statement would be',  # 'the statement would be [false]'; land 'be' (excl verdict); non-hypothetical occ (not the 'would be true' case)
    365: 'the statement that exactly one is in Uruguay is',  # hedged verdict 'likely false'; low-confidence row; land 'is' before 'likely' (earlier tentative occ walked back)
    366: 'none of them are in Yemen, which',  # 'which contradicts'; land 'which'
    396: 'that would mean the statement is',
    425: 'both are in Colombia, which',  # 'which isn't the case' (=false paraphrase); land 'which'
    428: 'one of them is in Benin is',
    465: 'exactly 0," which',  # match via 'which matches'; land 'which' (P=quote/paraphrase anchor)
    470: 'exactly 0 are in Kenya, which',  # 'which matches'; land 'which'
    491: 're-evaluation, the statement holds',  # heavy interpretive waffle ('exactly two'=='both'?); cut at final in-reasoning commit
    507: 'are in Ukraine, making the statement',  # 'making the statement [true/false]'; land on token before the verdict word
    521: 'zero are, making the statement',  # 'making the statement [true/false]'; land on token before the verdict word
    531: 'That means the statement is',  # paraphrase 'the statement is claiming [discrepancy]' (verdict embedded, no clean which-clause); land copula 'is'
    556: 'the number is two, which',  # 'which matches'; land 'which'
    571: 'That would mean the statement is',  # paraphrase 'the statement is claiming [discrepancy]' (verdict embedded, no clean which-clause); land copula 'is'
    574: 'only one is in Brazil, the statement would be',  # 'the statement would be [false]'; land 'be' (excl verdict); non-hypothetical occ (not the 'would be true' case)
}

filtered_F3_test = apply_manual_corrections(
    dataset=filtered_F3_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F3_filtered_test.csv",
)

Row 3: corrected.
Row 78: corrected.
Row 79: corrected.
Row 95: corrected.
Row 113: corrected.
Row 126: corrected.
Row 131: corrected.
Row 174: corrected.
Row 179: corrected.
Row 232: corrected.
Row 333: corrected.
Row 365: corrected.
Row 366: corrected.
Row 396: corrected.
Row 425: corrected.
Row 428: corrected.
Row 465: corrected.
Row 470: corrected.
Row 491: corrected.
Row 507: corrected.
Row 521: corrected.
Row 531: corrected.
Row 556: corrected.
Row 571: corrected.
Row 574: corrected.
Dropping 2 row(s): [7, 273]


In [ ]:
corrections = {
    11: 'are in Sudan would be',  # 'the statement would be [false/true]'; land 'be'; committed occ (not a hypothetical 'would be')
    19: 'them are in Tonga, which',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    70: 'correcting, the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    133: 'not 0. Therefore, the statement is',
    138: 'So the statement is',  # 'the statement is claiming/saying [discrepancy]' — verdict in SAME clause, no clean which-clause; land copula 'is'
    150: 'is zero. The statement',  # 'The statement correctly asserts' (correctly=accurately); land 'statement', excl 'correctly'
    164: 'count is 2. Therefore, the statement is',
    169: "Since there's only one, the statement is",
    176: 'then the statement is',
    207: 'drop',  # DROP - truncated mid-reasoning, no </think>
    212: 'in Mongolia, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    221: 'That means the statement is',  # 'the statement is claiming/saying [discrepancy]' — verdict in SAME clause, no clean which-clause; land copula 'is'
    223: 'exactly 0 are in South Korea is',
    228: 'exactly 2 are in Nigeria, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'; heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    240: 'drop',  # DROP - truncated mid-reasoning, no </think>
    246: 'saying exactly one is in Malaysia is',
    266: 'exactly 0 are in Uganda, which',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    315: "exactly 0 are in China, but that's",  # 'that's not the case' (=false paraphrase); keep subj+copula "that's", excl 'not the case'
    319: 'The statement must be',  # two-token 'must be true'; land 'be', excl 'true'; model geography error (wrong verdict/premise); cut fixes WHERE, not correctness — flag for competence gate
    383: 'exactly 0 are in Honduras is',
    401: 'of them are in Mexico, which',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    430: 'Therefore, the statement',  # 'The statement correctly asserts' (correctly=accurately); land 'statement', excl 'correctly'
    471: 'exactly two are in Timor-Leste is',
    494: 'indeed in Brazil. Therefore, the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    564: '"exactly 0," that',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    569: 'that would mean the statement is',
    621: 'exactly two are in Yemen, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    626: 'both are in India, so the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    636: 'in Jamaica, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    673: "in reality, it's 1. Therefore, the statement is",
    679: 'That means the statement is',  # 'the statement is claiming/saying [discrepancy]' — verdict in SAME clause, no clean which-clause; land copula 'is'
    711: 'are in Sudan, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    713: 'exactly one is in Cuba is',
    726: 'both are correct, making the statement',  # 'making the statement [true/false]'; land token before the verdict word
    732: 'drop',  # DROP - truncated mid-reasoning, no </think>
    747: '"Exactly 0," which',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    754: 'so exactly two are. Therefore, the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    763: 'then the statement would be',  # 'the statement would be [false/true]'; land 'be'; committed occ (not a hypothetical 'would be')
    787: 'of Argentina. The statement',  # 'The statement correctly asserts' (correctly=accurately); land 'statement', excl 'correctly'
    803: 'drop',  # DROP - truncated mid-reasoning, no </think>
    826: 'are in Pakistan, that would be',  # 'the statement would be [false/true]'; land 'be'; committed occ (not a hypothetical 'would be')
    829: 'That means the statement is',  # 'the statement is claiming/saying [discrepancy]' — verdict in SAME clause, no clean which-clause; land copula 'is'
    838: 'indeed in Saudi Arabia. That',  # 'That fits the statement' (=matches); land 'That'
    863: 'in China. That would make the statement',  # 'making the statement [true/false]'; land token before the verdict word
    882: 'are in the Dominican Republic, which',  # 'which contradicts'; land 'which'
    905: 'So that would mean the statement is',
    918: 'in Venezuela. So the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    946: 'in Barbados, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    972: 'saying exactly one is would be',  # 'the statement would be [false/true]'; land 'be'; committed occ (not a hypothetical 'would be'); model phrasing garbled ('saying exactly one is would be incorrect'); cut before the verdict
    980: 'indeed in Togo, the statement is',
    994: 'list is zero, making the statement',  # 'making the statement [true/false]'; land token before the verdict word
    1021: 'not 0. Therefore, the statement is',  # 'the statement is claiming/saying [discrepancy]' — verdict in SAME clause, no clean which-clause; land copula 'is'
    1043: 'of them are in Yemen, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    1159: 'Philippines), then the statement would be',  # 'the statement would be [false/true]'; land 'be'; committed occ (not a hypothetical 'would be'); model geography error (wrong verdict/premise); cut fixes WHERE, not correctness — flag for competence gate
    1174: 'yes, the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    1298: '(both) are in Mexico, making the statement',  # 'making the statement [true/false]'; land token before the verdict word
    1318: 'are in Malta, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    1326: '"exactly two," which',  # 'which/that matches'; land pronoun (quote-char rows use straight double-quotes)
    1332: 'listed are in Italy, which is',  # clean 'which is [true/correct]' clause after a claim-restatement; land 'is'
    1344: 'one city is in the Philippines, making the statement',  # 'making the statement [true/false]'; land token before the verdict word; reasoning complete + verdict present but NO 'Answer:' token emitted; usable for probe, unscoreable by competence gate
    1371: 'cities, so the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
    1382: 'should be that the statement is',  # heavy interpretive waffle ('exactly two'=='both'?); cut at FINAL in-reasoning commit
}

filtered_F3_train = apply_manual_corrections(
    dataset=filtered_F3_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F3_filtered_train.csv",
)

Row 11: corrected.
Row 19: corrected.
Row 70: corrected.
Row 133: corrected.
Row 138: corrected.
Row 150: corrected.
Row 164: corrected.
Row 169: corrected.
Row 176: corrected.
Row 212: corrected.
Row 221: corrected.
Row 223: corrected.
Row 228: corrected.
Row 246: corrected.
Row 266: corrected.
Row 315: corrected.
Row 319: corrected.
Row 383: corrected.
Row 401: corrected.
Row 430: corrected.
Row 471: corrected.
Row 494: corrected.
Row 564: corrected.
Row 569: corrected.
Row 621: corrected.
Row 626: corrected.
Row 636: corrected.
Row 673: corrected.
Row 679: corrected.
Row 711: corrected.
Row 713: corrected.
Row 726: corrected.
Row 747: corrected.
Row 754: corrected.
Row 763: corrected.
Row 787: corrected.
Row 826: corrected.
Row 829: corrected.
Row 838: corrected.
Row 863: corrected.
Row 882: corrected.
Row 905: corrected.
Row 918: corrected.
Row 946: corrected.
Row 972: corrected.
Row 980: corrected.
Row 994: corrected.
Row 1021: corrected.
Row 1043: corrected.
Row 1159: corrected.


#### Filtering F2

In [ ]:
filtered_F2_test = filter_dataset(dataset=F2_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F2_filtered_test.csv")

 20%|██        | 104/512 [19:57<54:02,  7.95s/it] 

Row 103: phrase found multiple times (ambiguous), skipping: 'part of the statement'


 25%|██▌       | 128/512 [23:09<46:46,  7.31s/it]  

Row 127: phrase found multiple times (ambiguous), skipping: 'part of the statement is'


 36%|███▌      | 182/512 [31:58<1:00:49, 11.06s/it]

...

 92%|█████████▏| 471/512 [1:15:42<07:33, 11.05s/it]

Row 470: phrase found multiple times (ambiguous), skipping: 'statement is'


 98%|█████████▊| 500/512 [1:22:59<02:43, 13.64s/it]

Checkpointed at row 500/512


100%|██████████| 512/512 [1:25:57<00:00, 10.07s/it]


In [ ]:
filtered_F2_train = filter_dataset(dataset=F2_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F2_filtered_train.csv")

  1%|          | 8/1194 [01:43<4:31:20, 13.73s/it]

Row 7: phrase found multiple times (ambiguous), skipping: 'the statement is'


  3%|▎         | 39/1194 [09:36<4:35:45, 14.32s/it]

Row 38: phrase found multiple times (ambiguous), skipping: 'which is'


  8%|▊         | 100/1194 [27:08<7:24:07, 24.36s/it]

...

 90%|█████████ | 1078/1194 [6:48:30<24:37, 12.73s/it]  

evaluate_sample attempt 1/3 failed: TypeError("'NoneType' object is not iterable")


 92%|█████████▏| 1100/1194 [6:54:22<29:28, 18.81s/it]

Row 1099: phrase found multiple times (ambiguous), skipping: 'the statement'


100%|██████████| 1194/1194 [7:19:52<00:00, 22.10s/it]


In [ ]:
# F2_filtered_test manual corrections - 8 filter-failed rows (5 fixed + 3 drops).
# F2 = two-way conjunction: 'It is the case both that [A] and that [B]'. True iff both conjuncts true.
# Cut: first DETERMINING whole-statement disclosure. False conjunction -> first conjunct declared FALSE
#   (a false conjunct fixes the whole). Cities' actual locations, incl. 'X is not in [claimed country]',
#   are KEPT as component facts (parallels F3 keeping the count); the explicit sub-part/whole verdict is
#   the disclosure. 3 DROPS: hedged rows where the model never commits in-reasoning (verdict only after </think>).
 
corrections = {
    103: 'drop',  # DROP - hedged: model never commits in-reasoning (unsure of Kalaburagi); conditional verdict only
    127: 'the first part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula (is), excl the verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    181: 'Since both cities are',  # TRUE conjunction: no single conjunct determines; cut at the both-true aggregation 'both cities are correctly identified' -> land 'are', excl 'correctly...' (evaluative aggregation).
    209: 'So the first part is',  # first conjunct declared FALSE = determining for the AND; land copula (is), excl the verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    226: 'drop',  # DROP - hedged: model never commits in-reasoning (unsure of Warri); conditional verdict only
    242: 'drop',  # DROP - hedged: model never commits in-reasoning (unsure of Pallabi); conditional verdict only
    428: 'since the first part is',  # first conjunct declared FALSE = determining for the AND; land copula (is), excl the verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    470: 'the first part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula (is), excl the verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
}

filtered_F2_test = apply_manual_corrections(
    dataset=filtered_F2_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F2_filtered_test.csv",
)

Row 127: corrected.
Row 181: corrected.
Row 209: corrected.
Row 428: corrected.
Row 470: corrected.
Dropping 3 row(s): [103, 226, 242]


In [ ]:
# F2_filtered_train manual corrections - 31 filter-failed rows (all fixed, no drops).
# F2 = two-way conjunction: 'It is the case both that [A] and that [B]'. True iff both conjuncts true.
# Cut: first DETERMINING whole-statement disclosure. False conjunction -> first conjunct declared FALSE.
#   Cities' actual locations, incl. 'X is not in [claimed country]', are KEPT as component facts
#   (parallels F3 keeping the count); the explicit sub-part/whole verdict is the disclosure.
#   True conjunction -> cut at the both-true aggregation.
 
corrections = {
    7: 'the second part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    38: 'in Iraq, which is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    150: 'so both cities are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    159: 'already makes the first part',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    177: 'Nigeria), and the second part is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    255: 'all together, both statements are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    259: 'should make the entire statement',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    300: 'the second part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model geography error: 300 Hamhung IS in North Korea; 466 Hoji ya Henda IS in Angola; 898 Malacca City exists (model pedantic state-vs-city). Cut fixes WHERE, not correctness.
    316: 'the first part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    339: 'the second part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). typo on first conjunct ('Xiayngh'->Xi'an); model waffles first part then resolves; cut at the clean second-conjunct-false verdict.
    377: 'the first part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    460: 'in China. The second part',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    466: 'However, the second part is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model geography error: 300 Hamhung IS in North Korea; 466 Hoji ya Henda IS in Angola; 898 Malacca City exists (model pedantic state-vs-city). Cut fixes WHERE, not correctness.
    476: 'make the first part of the statement',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    503: 'Since both parts are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    517: 'So both cities are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    570: 'The second part',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    619: 'Contagem is in China, which is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    655: 'Zhu Cheng City is in China (which is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    666: 'So, both cities are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    707: 'this part of the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    722: "the first part of the conjunction (the 'and') is",  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    738: 'the first part is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    747: 'Therefore, the statement is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model gives no clean per-conjunct verdict, jumps straight to whole-statement verdict; cut there.
    751: 'the statement says. That',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    898: 'The second part is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model geography error: 300 Hamhung IS in North Korea; 466 Hoji ya Henda IS in Angola; 898 Malacca City exists (model pedantic state-vs-city). Cut fixes WHERE, not correctness.
    1023: 'knowledge, the first part is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    1033: 'in India, that',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count). model uncertain about a city but DOES commit (assumption / 'seems' / 'definitely false'); kept, flagged for competence review.
    1046: 'Since both cities are',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
    1064: 'Ho Chi Minh City is in Nigeria is',  # first conjunct declared FALSE = determining for the AND; land copula, excl verdict word. Location facts incl. 'X is not in [claimed country]' kept as component facts (parallels F3 keeping the count).
    1099: 'together, the statement',  # TRUE conjunction: no single conjunct determines; cut at the first both-true aggregation ('both X are ...' / 'the entire statement is true'); land copula/'statement', excl the evaluative/verdict.
}

filtered_F2_train = apply_manual_corrections(
    dataset=filtered_F2_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F2_filtered_train.csv",
)

Row 7: corrected.
Row 38: corrected.
Row 150: corrected.
Row 159: corrected.
Row 177: corrected.
Row 255: corrected.
Row 259: corrected.
Row 300: corrected.
Row 316: corrected.
Row 339: corrected.
Row 377: corrected.
Row 460: corrected.
Row 466: corrected.
Row 476: corrected.
Row 503: corrected.
Row 517: corrected.
Row 570: corrected.
Row 619: corrected.
Row 655: corrected.
Row 666: corrected.
Row 707: corrected.
Row 722: corrected.
Row 738: corrected.
Row 747: corrected.
Row 751: corrected.
Row 898: corrected.
Row 1023: corrected.
Row 1033: corrected.
Row 1046: corrected.
Row 1064: corrected.
Row 1099: corrected.


#### Filtering F1

In [ ]:
filtered_F1_test = filter_dataset(dataset=F1_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F1_filtered_test.csv")

  0%|          | 2/512 [00:31<2:19:54, 16.46s/it]

Row 1: phrase not found verbatim, skipping: 'the statement is'


  8%|▊         | 41/512 [12:44<2:30:05, 19.12s/it]

Row 40: phrase found multiple times (ambiguous), skipping: 'not in Nepal is'


  8%|▊         | 42/512 [13:43<4:03:24, 31.07s/it]

...

 97%|█████████▋| 496/512 [1:57:26<03:26, 12.91s/it]

Row 495: phrase found multiple times (ambiguous), skipping: 'the statement is'


 97%|█████████▋| 499/512 [1:58:03<02:36, 12.01s/it]

Checkpointed at row 500/512


100%|██████████| 512/512 [2:01:36<00:00, 14.25s/it]


In [ ]:
filtered_F1_train = filter_dataset(dataset=F1_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F1_filtered_train.csv")

  0%|          | 5/1194 [01:03<3:39:40, 11.09s/it]

Row 4: phrase found multiple times (ambiguous), skipping: 'the statement is'


  1%|          | 13/1194 [02:59<4:06:23, 12.52s/it]

Row 12: phrase found multiple times (ambiguous), skipping: "claiming it's not in India is"


  7%|▋         | 81/1194 [17:27<4:56:26, 15.98s/it]

...

 94%|█████████▍| 1128/1194 [3:34:20<16:30, 15.01s/it]

Row 1127: phrase found multiple times (ambiguous), skipping: 'Therefore, the statement is'


 97%|█████████▋| 1158/1194 [3:40:22<07:26, 12.39s/it]

Row 1157: phrase found multiple times (ambiguous), skipping: 'Therefore, the statement is'


100%|██████████| 1194/1194 [3:48:24<00:00, 11.48s/it]


In [ ]:
# F1_filtered_test manual corrections - 16 filter-failed rows (15 fixed + 1 drop).
# F1 = negation: 'The city of X is not in country Y'. TRUE iff X is genuinely elsewhere; FALSE iff X is in Y.
# Cut: first determining whole-statement disclosure = the truth-verdict on the negated statement.
#   The claim-restatement 'it's not in Y' (the statement's own content) is KEPT; the verdict word is excluded.
#   Hypothetical 'would be false' with an unestablished antecedent is NOT a disclosure (kept); cut at the
#   committed verdict. 1 DROP: row 1 truncated (no </think>).

corrections = {
    1: 'drop',  # DROP - truncated mid-reasoning, no </think>, no verdict
    40: "would mean the statement claiming it's not in Nepal is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    41: 'in China. Therefore, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count). model geography error: 41 Panshan IS in China (Liaoning) - model puts it in South Korea and concludes True. Cut fixes WHERE, not correctness.
    78: "it's not there would be",  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
    118: 'as being in China, then the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count). statement name near-typo (Xianyang vs Xi'an); model waffles then resolves both readings to 'in China -> false'; cut at first committed verdict.
    162: "the statement saying it's not in Lebanon is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    204: "it's not there would be",  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
    207: "the statement that it's not there is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    291: "it's not in Uganda would be",  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
    361: 'this would mean the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    381: 'indeed in China, making the original statement',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count). model never firmly certain of the city (leans 'in China'); cut at first committed directional verdict, skipping earlier hedged 'would be false'.
    382: 'that would mean the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    437: "since it's in China, the statement is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect'); land copula, excl verdict word. The claim-restatement 'it's not in Y' is kept (statement's own content, like F3 keeping the count).
    446: "it's not in the Philippines would be",  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
    472: 'then this statement would be',  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
    495: "claiming it's not would be",  # 'the statement ... would be false/correct' with the city's location already established; land 'be', excl verdict word.
}

filtered_F1_test = apply_manual_corrections(
    dataset=filtered_F1_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F1_filtered_test.csv",
)

Row 40: corrected.
Row 41: corrected.
Row 78: corrected.
Row 118: corrected.
Row 162: corrected.
Row 204: corrected.
Row 207: corrected.
Row 291: corrected.
Row 361: corrected.
Row 381: corrected.
Row 382: corrected.
Row 437: corrected.
Row 446: corrected.
Row 472: corrected.
Row 495: corrected.
Dropping 1 row(s): [1]


In [ ]:
# F1_filtered_train manual corrections - 25 filter-failed rows (24 fixed + 1 drop).
# F1 = negation: 'The city of X is not in country Y'. TRUE iff X is genuinely elsewhere; FALSE iff X is in Y.
# Cut: first determining whole-statement verdict on the negated statement. The claim-restatement 'it's not
#   in Y' (statement's own content) is KEPT; verdict word excluded. Hedged/hypothetical 'would be' with an
#   unestablished antecedent is NOT a disclosure (kept); cut at the committed verdict.
# 1 DROP: row 940 renders no explicit verdict (only restates the claim's location).

corrections = {
    4: 'Therefore, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    12: "the original statement claiming it's not in India is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    80: 'Lilongwe is not in Cameroon is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    144: 'in Japan, making the statement',  # 'making the statement false'; land 'statement', excl 'false' (final committed occ after a waffle). model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    341: 'located in Brazil. Therefore, the statement',  # 'the statement contradicts the known facts'; land 'statement', excl 'contradicts'.
    362: 'not in Angola would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.)
    450: 'not in China would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.) waffle (450 Taiwan-vs-China status; 842 Ordos/Erdos China-vs-Mongolia); leans one way, cut at the committed verdict.
    475: 'not in Ghana must be',  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    574: "isn't in Jordan, which is",  # clean 'which is [absolutely] true' after a claim-restatement; land 'is'.
    646: 'the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    647: "claiming it's not in China must be",  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    728: 'Santiago de Queretaro is not in Indonesia is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    763: 'likely that the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    814: "claiming it's not in Pakistan is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    842: 'claiming that Ordos is not in China would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.) waffle (450 Taiwan-vs-China status; 842 Ordos/Erdos China-vs-Mongolia); leans one way, cut at the committed verdict.
    844: 'the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    859: 'Thane is not in Bangladesh is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    861: 'Astana is not in Kazakhstan is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    940: 'drop',  # DROP - no explicit verdict in reasoning - model only restates the claim's locational content ('Mombasa is not in Brazil') with no true/false/correct/incorrect; nothing to cut before that isn't lexically the claim
    1058: 'the original statement must be',  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    1059: 'not Russia, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    1064: 'must be in Japan. Therefore, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1086: 'That would mean the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1127: 'That would mean the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1157: 'then this statement would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.)
}

filtered_F1_train = apply_manual_corrections(
    dataset=filtered_F1_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F1_filtered_train.csv",
)

Row 4: corrected.
Row 12: corrected.
Row 80: corrected.
Row 144: corrected.
Row 341: corrected.
Row 362: corrected.
Row 450: corrected.
Row 475: corrected.
Row 574: corrected.
Row 646: corrected.
Row 647: corrected.
Row 728: corrected.
Row 763: corrected.
Row 814: corrected.
Row 842: corrected.
Row 844: corrected.
Row 859: corrected.
Row 861: corrected.
Row 1058: corrected.
Row 1059: corrected.
Row 1064: corrected.
Row 1086: corrected.
Row 1127: corrected.
Row 1157: corrected.
Dropping 1 row(s): [940]


#### Filtering F0

In [ ]:
filtered_F0_test = filter_dataset(dataset=F0_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F0_filtered_test.csv")

  1%|          | 6/512 [01:04<1:38:30, 11.68s/it]

Row 5: phrase found multiple times (ambiguous), skipping: 'Therefore, the statement is'


  6%|▌         | 29/512 [04:55<1:37:08, 12.07s/it]

Row 28: phrase found multiple times (ambiguous), skipping: "idea that Riga isn't"


 19%|█▉        | 99/512 [16:15<1:06:12,  9.62s/it]

...

 83%|████████▎ | 425/512 [1:10:45<14:08,  9.75s/it]

Row 424: phrase not found verbatim, skipping: 'in China. That doesn\x00'


 97%|█████████▋| 499/512 [1:22:20<02:03,  9.51s/it]

Checkpointed at row 500/512


100%|██████████| 512/512 [1:24:05<00:00,  9.85s/it]


In [ ]:
filtered_F0_train = filter_dataset(dataset=F0_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/F0_filtered_train.csv")

  1%|          | 10/1194 [01:45<3:24:24, 10.36s/it]

Row 9: phrase found multiple times (ambiguous), skipping: 'the statement'


  8%|▊         | 92/1194 [15:15<3:18:20, 10.80s/it]

Row 91: phrase not found verbatim, skipping: 'Conclusion: The statement is'


  8%|▊         | 99/1194 [16:27<3:15:25, 10.71s/it]

...

 95%|█████████▍| 1132/1194 [3:24:24<17:06, 16.56s/it]

Row 1131: phrase not found verbatim, skipping: 'The statement is'


 96%|█████████▌| 1144/1194 [3:27:12<11:43, 14.08s/it]

Row 1143: phrase found multiple times (ambiguous), skipping: 'the statement is'


100%|██████████| 1194/1194 [3:38:01<00:00, 10.96s/it]


In [ ]:
# F0_filtered_test manual corrections - 4 filter-failed rows (all fixed, no drops).
# F0 = base positive task: 'The city of X is in country Y'. TRUE iff X is in Y.
# Cut: first determining whole-statement verdict; land copula, excl verdict word. Location facts
#   (incl. 'X is in Z, not Y') kept as component facts.
 
corrections = {
    5: 'upon reflection, the statement',  # district-vs-city waffle (false->true); cut at final in-reasoning commit 'the statement correctly identifies...' (later 'is true' is the post-</think> answer).
    28: 'the original statement claiming Riga is in Brazil is',  # explicit verdict; land copula, excl verdict word. Location facts and claim-negations ('Riga is not in Brazil') kept.
    255: 'in the UAE must be',  # two-token 'must be incorrect'; land 'be', excl verdict.
    424: "says it's in China. That",  # informal rejection 'That doesn't make sense' (re: the claim); land 'That', excl the complement. (verifier hit a null-byte in "doesn't".)
}

filtered_F0_test = apply_manual_corrections(
    dataset=filtered_F0_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F0_filtered_test.csv",
)

Row 5: corrected.
Row 28: corrected.
Row 255: corrected.
Row 424: corrected.


In [ ]:
# F0_filtered_train manual corrections - 20 filter-failed rows (13 fixed + 7 drops).
# F0 = base positive task: 'The city of X is in country Y'. TRUE iff X is in Y.
# Cut: first determining whole-statement verdict; land copula, excl verdict word. Location facts kept.
# 7 DROPS are CONFABULATIONS - the model did not evaluate the given statement (hallucinated an
#   arithmetic conjunct, misread the city name, or swapped the country). 4 flip the label
#   (9,400,524,1143); 3 preserve it (91,889,1058). Flagged for the faithfulness / competence analysis.

corrections = {
    9: 'drop',  # DROP - CONFABULATION (label mismatch): prompt 'Sevastopol is in Ukraine' (true), model evaluated 'Sevastopol in the United States' -> false
    91: 'drop',  # DROP - CONFABULATION (label match): prompt 'Meerut is in India', model evaluated 'Meerut in the northern part of India'; no explicit verdict
    101: 'Yongzhou is in the Philippines is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    258: 'all together, the statement',  # 'the statement correctly identifies ...'; land 'statement', excl 'correctly'.
    271: 'in Japan would be',  # 'would be incorrect' with location established; land 'be'.
    319: 'conclude that Orumiyeh is',  # model followed the 'no verdict words' instruction; conclusion is a bare claim-affirmation 'Orumiyeh is [in Iran]'; land copula 'is', excl the affirmed claim content.
    382: 'probable that the statement is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    400: 'drop',  # DROP - CONFABULATION (label mismatch): misread Agege->Ageo; evaluated 'Ageo in Japan' (true) but Agege is in Nigeria -> given statement is false
    445: 'I think the statement is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept. hedged 'likely true' but explicit; land 'is', excl 'likely'.
    457: 'indeed Baku. So the statement',  # 'the statement correctly identifies ...'; land 'statement', excl 'correctly'.
    504: 'Mukim Pulai is in Malaysia is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    524: 'drop',  # DROP - CONFABULATION (label mismatch): prompt 'Shangyu is in China' (true), model hallucinated an arithmetic conjunct 'sqrt2+sqrt3>4' -> whole false
    610: 'Burkina Faso, that',  # informal rejection 'that doesn't sound right' (re: the claim); land 'that', excl complement. (verifier hit a control-char in "doesn't".)
    834: 'That would mean the statement is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    889: 'drop',  # DROP - CONFABULATION (label match): misread 'Dushanbe'->'Dushan be'; evaluated 'Dushan in China' instead of 'Dushanbe in China'
    899: 'Shaoxing is in South Korea is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    995: 'in Malaysia. Therefore, the statement is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept.
    1058: 'drop',  # DROP - CONFABULATION (label match): prompt 'Situbondo is in Indonesia' (true), model hallucinated coords + arithmetic conjunct
    1131: 'Teni is a city in India is',  # explicit verdict; land copula, excl verdict word. Location facts / claim-restatements kept. model geography error: Theni IS a district in Tamil Nadu, India, so the statement is actually true; model calls it false. Cut fixes WHERE, not correctness.
    1143: 'drop',  # DROP - CONFABULATION (label mismatch): prompt 'Fuxin is in China' (true), model hallucinated arithmetic conjunct 'sqrt2+sqrt3>4' -> whole false
}

filtered_F0_train = apply_manual_corrections(
    dataset=filtered_F0_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/F0_filtered_train.csv",
)

Row 101: corrected.
Row 258: corrected.
Row 271: corrected.
Row 319: corrected.
Row 382: corrected.
Row 445: corrected.
Row 457: corrected.
Row 504: corrected.
Row 610: corrected.
Row 834: corrected.
Row 899: corrected.
Row 995: corrected.
Row 1131: corrected.
Dropping 7 row(s): [9, 91, 400, 524, 889, 1058, 1143]


#### Filtering A3

In [35]:
filtered_A3_test = filter_dataset(dataset=A3_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/A3_filtered_test.csv")

  0%|          | 0/300 [00:01<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
filtered_A3_train = filter_dataset(dataset=A3_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/A3_filtered_train.csv")

  1%|          | 7/700 [00:57<1:42:20,  8.86s/it]

Row 6: phrase not found verbatim, skipping: "Hmm, there's"


  4%|▎         | 25/700 [04:07<2:35:47, 13.85s/it]

Row 24: phrase found multiple times (ambiguous), skipping: 'So that'


  5%|▍         | 34/700 [05:30<2:17:08, 12.36s/it]

...

 95%|█████████▌| 665/700 [2:06:37<05:36,  9.60s/it]

Row 664: phrase not found verbatim, skipping: 'Hmm, that doesn\x19t'


100%|█████████▉| 699/700 [2:12:03<00:09,  9.25s/it]

Checkpointed at row 700/700


100%|██████████| 700/700 [2:12:12<00:00, 11.33s/it]


In [9]:
filtered_A3_test = pd.read_csv("../CoT_datasets/filtered/A3_filtered_test.csv")
filtered_A3_train = pd.read_csv("../CoT_datasets/filtered/A3_filtered_train.csv")

In [10]:
# A3_filtered_test manual corrections - 11 filter-failed rows (all fixed, no drops).
# A3 = equation verification: '(A op B) op (C op D) = result'. TRUE iff the LHS computes to the RHS.
# Cut: keep the arithmetic computation (the LHS value); cut at the COMPARISON to the RHS - the point
#   the model declares match / mismatch ('which matches', 'That doesn't match', 'the equation is off').
 
corrections = {
    45: 'is 37, which',  # computed LHS matches RHS: 'which/That matches the right side'; land pronoun, excl 'matches'. The computed value is kept.
    53: 'becomes -57. That',  # computed LHS matches RHS: 'which/That matches the right side'; land pronoun, excl 'matches'. The computed value is kept.
    59: 'would be -72. That',  # computed LHS matches RHS: 'which/That matches the right side'; land pronoun, excl 'matches'. The computed value is kept.
    85: 'That means the equation is',  # 'the equation is off by 2' (=LHS!=RHS); land 'is', excl 'off...'. (Earlier arithmetic slip 2472 self-corrected to 2850 before this point.)
    113: 'says it equals -18. That',  # computed LHS != RHS: informal 'That doesn't match/add up'; land 'That'/'that', excl 'doesn't...'. (verifier hit curly/control-char apostrophes in "doesn't".)
    131: '-182. That',  # computed LHS != RHS: informal 'That doesn't match/add up'; land 'That'/'that', excl 'doesn't...'. (verifier hit curly/control-char apostrophes in "doesn't".)
    168: 'equals -52. Hmm, that',  # computed LHS != RHS: informal 'That doesn't match/add up'; land 'That'/'that', excl 'doesn't...'. (verifier hit curly/control-char apostrophes in "doesn't".)
    172: 'equals 53. That',  # computed LHS != RHS: informal 'That doesn't match/add up'; land 'That'/'that', excl 'doesn't...'. (verifier hit curly/control-char apostrophes in "doesn't".)
    205: 'equals 304. That',  # computed LHS != RHS: informal 'That doesn't match/add up'; land 'That'/'that', excl 'doesn't...'. (verifier hit curly/control-char apostrophes in "doesn't".)
    245: 'brings us down to 286',  # computation ends at LHS=286; cut before the affirmation 'So yes, 315-29 does equal 286'; land the computed '286'.
    266: 'gives -385. That',  # computed LHS matches RHS: 'which/That matches the right side'; land pronoun, excl 'matches'. The computed value is kept.
}

filtered_A3_test = apply_manual_corrections(
    dataset=filtered_A3_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A3_filtered_test.csv",
)

Row 45: corrected.
Row 53: corrected.
Row 59: corrected.
Row 85: corrected.
Row 113: corrected.
Row 131: corrected.
Row 168: corrected.
Row 172: corrected.
Row 205: corrected.
Row 245: corrected.
Row 266: corrected.


In [11]:
# A3_filtered_train manual corrections - 16 filter-failed rows (all fixed, no drops).
# A3 = equation verification: '(A op B) op (C op D) = result'. TRUE iff LHS computes to RHS.
# Cut: keep the arithmetic computation; cut at the comparison to the RHS (match / mismatch declaration).
# Nearly all failed on the verifier's verbatim search because the 'doesn't' apostrophe is a curly quote
#   or control byte (\x19/\x00); landing on the pronoun before it sidesteps the encoding.

corrections = {
    6: "that means there's",  # 'that means there's a discrepancy' (=LHS!=RHS); keep subj+copula "there's", excl 'a discrepancy'.
    24: '6120. So that',  # computed LHS matches RHS: 'So that/That/which matches'; land pronoun, excl 'matches'. Computed value kept.
    33: 'equals 557. That',  # computed LHS matches RHS: 'So that/That/which matches'; land pronoun, excl 'matches'. Computed value kept.
    39: 'equals -190. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    75: 'sum equals 71, which',  # computed LHS matches RHS: 'So that/That/which matches'; land pronoun, excl 'matches'. Computed value kept.
    203: 'indeed 33. That',  # computed LHS matches RHS: 'So that/That/which matches'; land pronoun, excl 'matches'. Computed value kept.
    275: 'equals 210. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    348: 'equals -12. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    427: 'result is 16. Hmm, that',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    436: '-86. Wait, that',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    497: 'equals 61. Hmm, that',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    547: 'equals -46. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    574: 'equals 11. Hmm, that',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    591: 'equals 79. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    616: 'equals 19. That',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
    664: 'equals 87. Hmm, that',  # computed LHS != RHS: informal 'That/that doesn't match/add up'; land the pronoun, excl 'doesn't...'. (verifier missed these on curly/control-char apostrophes - \x19, \x00, curly.)
}

filtered_A3_train = apply_manual_corrections(
    dataset=filtered_A3_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A3_filtered_train.csv",
)

Row 6: corrected.
Row 24: corrected.
Row 33: corrected.
Row 39: corrected.
Row 75: corrected.
Row 203: corrected.
Row 275: corrected.
Row 348: corrected.
Row 427: corrected.
Row 436: corrected.
Row 497: corrected.
Row 547: corrected.
Row 574: corrected.
Row 591: corrected.
Row 616: corrected.
Row 664: corrected.


#### Filtering A2

In [12]:
filtered_A2_test = filter_dataset(dataset=A2_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/A2_filtered_test.csv")

 31%|███       | 92/300 [12:43<25:03,  7.23s/it] 

Row 91: phrase found multiple times (ambiguous), skipping: '17, which'


 33%|███▎      | 99/300 [13:49<29:58,  8.95s/it]

Checkpointed at row 100/300


 49%|████▉     | 147/300 [21:05<20:32,  8.06s/it]

...

 71%|███████   | 212/300 [30:13<11:38,  7.94s/it]

Row 211: phrase not found verbatim, skipping: 'but that doesn\x19t'


100%|█████████▉| 299/300 [43:15<00:07,  7.91s/it]

Checkpointed at row 300/300


100%|██████████| 300/300 [43:20<00:00,  8.67s/it]


In [13]:
filtered_A2_train = filter_dataset(dataset=A2_train, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/A2_filtered_train.csv")

  4%|▍         | 27/700 [04:07<2:02:30, 10.92s/it]

Row 26: phrase found multiple times (ambiguous), skipping: 'the statement is'


  6%|▌         | 43/700 [06:38<2:03:10, 11.25s/it]

Row 42: phrase found multiple times (ambiguous), skipping: 'That means the statement is'


 11%|█         | 74/700 [11:00<1:39:25,  9.53s/it]

...

 98%|█████████▊| 687/700 [1:42:55<02:23, 11.06s/it]

Row 686: phrase not found verbatim, skipping: "claims it's 11,623. There's"


100%|█████████▉| 699/700 [1:45:19<00:09,  9.41s/it]

Checkpointed at row 700/700


100%|██████████| 700/700 [1:45:27<00:00,  9.04s/it]


In [15]:
# A2_filtered_test manual corrections - 3 filter-failed rows (all fixed, no drops).
# A2 = equation verification '(A op B) op C = result'. Keep computation, cut at comparison to RHS.
 
corrections = {
    91: 'equals 17, which',  # 'which matches'; land 'which', excl 'matches'. Computed value kept.
    146: 'equals 51. That',  # 'That doesn't match up'; land 'That'/'that', excl 'doesn't...' (manual standard = drop the negation). NOTE: this is exactly the o3-vs-manual convention under review - if you settle on o3's convention, land on 'doesn't' instead.
    211: '816. Wait, but that',  # 'That doesn't match up'; land 'That'/'that', excl 'doesn't...' (manual standard = drop the negation). NOTE: this is exactly the o3-vs-manual convention under review - if you settle on o3's convention, land on 'doesn't' instead. model mis-multiplies then self-corrects to 816 (twice); first comparison 'that doesn't match' is at the first 816 - cut there.
}

filtered_A2_test = apply_manual_corrections(
    dataset=filtered_A2_test,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A2_filtered_test.csv",
)

Row 91: corrected.
Row 146: corrected.
Row 211: corrected.


In [ ]:
# F1_filtered_train manual corrections - 25 filter-failed rows (24 fixed + 1 drop).
# F1 = negation: 'The city of X is not in country Y'. TRUE iff X is genuinely elsewhere; FALSE iff X is in Y.
# Cut: first determining whole-statement verdict on the negated statement. The claim-restatement 'it's not
#   in Y' (statement's own content) is KEPT; verdict word excluded. Hedged/hypothetical 'would be' with an
#   unestablished antecedent is NOT a disclosure (kept); cut at the committed verdict.
# 1 DROP: row 940 renders no explicit verdict (only restates the claim's location).

corrections = {
    4: 'Therefore, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    12: "the original statement claiming it's not in India is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    80: 'Lilongwe is not in Cameroon is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    144: 'in Japan, making the statement',  # 'making the statement false'; land 'statement', excl 'false' (final committed occ after a waffle). model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    341: 'located in Brazil. Therefore, the statement',  # 'the statement contradicts the known facts'; land 'statement', excl 'contradicts'.
    362: 'not in Angola would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.)
    450: 'not in China would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.) waffle (450 Taiwan-vs-China status; 842 Ordos/Erdos China-vs-Mongolia); leans one way, cut at the committed verdict.
    475: 'not in Ghana must be',  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    574: "isn't in Jordan, which is",  # clean 'which is [absolutely] true' after a claim-restatement; land 'is'.
    646: 'the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    647: "claiming it's not in China must be",  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    728: 'Santiago de Queretaro is not in Indonesia is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    763: 'likely that the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    814: "claiming it's not in Pakistan is",  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    842: 'claiming that Ordos is not in China would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.) waffle (450 Taiwan-vs-China status; 842 Ordos/Erdos China-vs-Mongolia); leans one way, cut at the committed verdict.
    844: 'the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model uncertain but DOES commit; cut at the first committed verdict, skipping earlier hedged/hypothetical 'would be'.
    859: 'Thane is not in Bangladesh is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    861: 'Astana is not in Kazakhstan is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    940: 'drop',  # DROP - no explicit verdict in reasoning - model only restates the claim's locational content ('Mombasa is not in Brazil') with no true/false/correct/incorrect; nothing to cut before that isn't lexically the claim
    1058: 'the original statement must be',  # two-token 'must be true/false/incorrect'; land 'be', excl verdict.
    1059: 'not Russia, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict. model geography/logic error: 4 Huaihua IS in China; 144 Adachi IS a Tokyo ward; 646 negation-logic slip (thinks Nansana in Nigeria -> 'not in Uganda' TRUE, but writes 'false'); 814 Chunian IS in Pakistan; 1059 Kalininskiy-in-Transnistria is dubious. Cut fixes WHERE, not correctness.
    1064: 'must be in Japan. Therefore, the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1086: 'That would mean the statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1127: 'That would mean the original statement is',  # explicit verdict on the negated statement ('the statement (claiming it's not in Y) is true/false/incorrect/correct'); land copula, excl verdict.
    1157: 'then this statement would be',  # 'would be false/incorrect' with the city's location already established; land 'be'. (Where a first 'would be' was still hedged/hypothetical, cut moved to the committed verdict.)
}

filtered_A2_train = apply_manual_corrections(
    dataset=filtered_A2_train,
    corrections=corrections,
    tokenizer=tokenizer,
    dataset_file_path="../CoT_datasets/filtered/A2_filtered_train.csv",
)

Row 4: corrected.
Row 12: corrected.
Row 80: corrected.
Row 144: corrected.
Row 341: corrected.
Row 362: corrected.
Row 450: corrected.
Row 475: corrected.
Row 574: corrected.
Row 646: corrected.
Row 647: corrected.
Row 728: corrected.
Row 763: corrected.
Row 814: corrected.
Row 842: corrected.
Row 844: corrected.
Row 859: corrected.
Row 861: corrected.
Row 1058: corrected.
Row 1059: corrected.
Row 1064: corrected.
Row 1086: corrected.
Row 1127: corrected.
Row 1157: corrected.
Dropping 1 row(s): [940]


#### Filtering A1_test

In [16]:
filtered_A1_test = filter_dataset(dataset=A1_test, tokenizer=tokenizer, evaluator_model="openai/o3", dataset_file_path="../CoT_datasets/filtered/A1_filtered_test.csv")

  0%|          | 1/300 [00:14<1:11:48, 14.41s/it]

Row 0: phrase found multiple times (ambiguous), skipping: 'that the equation is'


  4%|▎         | 11/300 [02:36<1:20:39, 16.74s/it]

Row 10: phrase not found verbatim, skipping: 'The statement is'


  6%|▌         | 17/300 [03:54<1:09:39, 14.77s/it]

...

 98%|█████████▊| 293/300 [1:02:28<01:30, 12.94s/it]

Row 292: phrase found multiple times (ambiguous), skipping: 'the statement is'


100%|█████████▉| 299/300 [1:03:42<00:12, 12.79s/it]

Checkpointed at row 300/300


100%|██████████| 300/300 [1:03:50<00:00, 12.77s/it]
